### Read from raw volume

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

providers_schema = StructType([
    StructField("provider_id", StringType(), False),
    StructField("doctor_name", StringType(), True),
    StructField("specialty", StringType(), True),
    StructField("location", StringType(), True),
])

df_raw = spark.read.csv(
    "/Volumes/workspace/default/myvol/raw/providers/providers_1000.csv",
    header=True,
    schema=providers_schema
)
print(f"Raw rows: {df_raw.count()}")
df_raw.show(5)

### Add metadata columns

In [0]:
df_bronze = df_raw \
    .withColumn("_ingestion_timestamp", F.current_timestamp()) \
    .withColumn("_source_file", F.lit("providers_1000.csv")) \
    .withColumn("_source_layer", F.lit("bronze")) \
    .withColumn("_is_phi", F.lit(False))

print(f"Bronze rows: {df_bronze.count()}")

### Write to Delta

In [0]:
df_bronze.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/default/myvol/bronze/providers/")

print("✅ Bronze providers Delta table written")

### Validate

In [0]:
df_verify = spark.read.format("delta") \
    .load("/Volumes/workspace/default/myvol/bronze/providers/")
assert df_verify.count() == df_raw.count(), "❌ Row count mismatch!"
print(f"✅ Validation passed — {df_verify.count()} rows")